# Yuda 8 — All Improvements
C_focal v2 (20ep+label smooth) + ConvNeXt V2 + Stacking all models

In [1]:
import sys
sys.path.insert(0, "../..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import gc
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from torch.utils.data import Dataset, DataLoader

import open_clip
import warnings
warnings.filterwarnings('ignore')

from modules.utils.config import *
from modules.utils.seed import set_seed
from modules.utils.load_data import load_train
from modules.models.factory import TrashClassifier
from modules.data.dataset import get_dataloaders, get_transforms, TrashDataset

set_seed(SEED)
print(f"Device: {DEVICE}")
print(f"open_clip: {open_clip.__version__}")

/home/prayudahlah/projects/external/big-data-big-trouble/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
open_clip: 3.3.0


In [2]:
_CLASS_TO_IDX = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ['Recyclable', 'Electronic', 'Organic']

CLIP_NAME = 'ViT-B-32'
CLIP_PRETRAINED = 'laion2b_s34b_b79k'
clip_model, _, clip_transform = open_clip.create_model_and_transforms(
    CLIP_NAME, pretrained=CLIP_PRETRAINED
)
clip_tokenizer = open_clip.get_tokenizer(CLIP_NAME)
clip_model = clip_model.to('cpu')
clip_model.eval()
clip_visual = clip_model.visual.to(DEVICE)
clip_dim = clip_visual.output_dim
print(f"CLIP dim: {clip_dim}")

prompts_improved = [
    "a photo of recyclable items like paper, plastic, glass, and metal",
    "a photo of electronic waste like computers, phones, cables, and batteries",
    "a photo of organic waste like food scraps, leaves, and plants",
]

class CLIPWithTextFeatures(nn.Module):
    def __init__(self, clip_model, clip_visual, prompts, num_classes=3):
        super().__init__()
        self.visual = clip_visual
        self.encoder = self.visual
        self.logit_scale = clip_model.logit_scale
        dim = self.visual.output_dim
        text_tokens = clip_tokenizer(prompts)
        with torch.no_grad():
            text_feat = clip_model.encode_text(text_tokens)
            text_feat = text_feat / text_feat.norm(dim=-1, keepdim=True)
        self.register_buffer('text_features', text_feat)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(dim + 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        v = self.visual(x)
        v_norm = v / v.norm(dim=-1, keepdim=True)
        sim = v_norm @ self.text_features.T
        combined = torch.cat([v, sim * self.logit_scale.exp()], dim=1)
        return self.classifier(combined)
    def freeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = False
    def unfreeze_encoder(self):
        for p in self.visual.parameters():
            p.requires_grad = True


def get_probs(model, loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for x, _ in tqdm(loader, desc="Inference", leave=False):
            x = x.to(DEVICE)
            all_probs.append(torch.softmax(model(x), dim=1).cpu().numpy())
    return np.concatenate(all_probs)


def train_head_only(model, train_loader, val_loader, name, epochs=10, criterion=None):
    model = model.to(DEVICE)
    model.freeze_encoder()
    criterion = criterion or nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        for x, y in tqdm(train_loader, desc=f"  E{epoch+1}", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            opt.step()
        sched.step()
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds.extend(model(x).argmax(dim=1).cpu().numpy())
                labs.extend(y.cpu().numpy())
        f1 = f1_score(labs, preds, average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_state = model.state_dict()
        print(f"  E{epoch+1}: val_f1={f1:.4f} (best={best_f1:.4f})")
    model.load_state_dict(best_state)
    result = {'name': name, 'best_val_f1': best_f1}
    torch.save(best_state, RESULTS / f'{name}.pt')
    json.dump(result, open(RESULTS / f'{name}.json', 'w'))
    return model, best_f1

conv_val_tfm = get_transforms(augment=False, img_size=224)
conv_val_tfm_300 = get_transforms(augment=False, img_size=300)
conv_train_tfm = get_transforms(augment=True, img_size=224)

CLIP dim: 512


---
## Dataloaders

In [3]:
train_loader, val_loader, val_ds = get_dataloaders(batch_size=32, num_workers=4)
df_train = train_loader.dataset.df
df_val = val_loader.dataset.df
print(f"Train: {len(df_train)} | Val: {len(df_val)}")

class SingleTransformDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(PROJECT_ROOT / row['path']).convert('RGB')
        return self.transform(img), _CLASS_TO_IDX[row['label']]

clip_val_ds = SingleTransformDataset(df_val, clip_transform)
clip_val_loader = DataLoader(clip_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

clip_train_ds = SingleTransformDataset(df_train, clip_transform)
clip_train_loader = DataLoader(clip_train_ds, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)

conv_val_ds = SingleTransformDataset(df_val, conv_val_tfm)
conv_val_loader = DataLoader(conv_val_ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

conv_val_ds_300 = SingleTransformDataset(df_val, conv_val_tfm_300)
conv_val_loader_300 = DataLoader(conv_val_ds_300, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

val_labels = np.array([_CLASS_TO_IDX[r['label']] for _, r in df_val.iterrows()])
print(f"Val labels: {len(val_labels)}")

Train: 21221 | Val: 5306
Val labels: 5306


---
## Phase 1 — Train C_focal v2 (20 epoch + Label Smoothing)

In [4]:
print("=" * 60)
print("Phase 1: C_focal v2 — 20 epoch + Label Smoothing")
print("=" * 60)

criterion_ls = nn.CrossEntropyLoss(label_smoothing=0.1)

m_cv2 = CLIPWithTextFeatures(clip_model, clip_visual, prompts_improved)
m_cv2, f1_cv2 = train_head_only(
    m_cv2, clip_train_loader, clip_val_loader,
    'yuda8_C_focal_v2', epochs=20, criterion=criterion_ls
)
print(f"C_focal v2 (label_smooth=0.1, 20ep): {f1_cv2:.4f}")

c_probs_cv2 = get_probs(m_cv2, clip_val_loader)
del m_cv2
torch.cuda.empty_cache()
gc.collect()

Phase 1: C_focal v2 — 20 epoch + Label Smoothing


  E1: val_f1=0.9723 (best=0.9723)


  E2: val_f1=0.9784 (best=0.9784)


  E3: val_f1=0.9799 (best=0.9799)


  E4: val_f1=0.9805 (best=0.9805)


...

  E17: val_f1=0.9835 (best=0.9839)


  E18: val_f1=0.9838 (best=0.9839)


  E19: val_f1=0.9838 (best=0.9839)


  E20: val_f1=0.9838 (best=0.9839)
C_focal v2 (label_smooth=0.1, 20ep): 0.9839


9

---
## Phase 2 — Train ConvNeXt V2 Tiny

In [5]:
from modules.training.train import fit

print("=" * 60)
print("Phase 2: ConvNeXt V2 Tiny (head-only)" )
print("=" * 60)

conv_train_loader_v2, conv_val_loader_v2, _ = get_dataloaders(
    batch_size=32, num_workers=4, img_size=224
)

model_conv_v2, f1_conv_v2 = train_head_only(
    TrashClassifier('convnextv2_tiny', num_classes=3),
    conv_train_loader_v2,
    conv_val_loader_v2,
    'yuda8_convnextv2_v2',
    epochs=15,
)
print(f"ConvNeXt V2: {f1_conv_v2:.4f}")

conv_v2_probs = get_probs(model_conv_v2, conv_val_loader)
del model_conv_v2
torch.cuda.empty_cache()
gc.collect()
print(f"ConvNeXt V2 probs: {conv_v2_probs.shape}")

Phase 2: ConvNeXt V2 Tiny (head-only)


  E1: val_f1=0.9623 (best=0.9623)


  E2: val_f1=0.9684 (best=0.9684)


  E3: val_f1=0.9688 (best=0.9688)


  E4: val_f1=0.9707 (best=0.9707)


...

  E12: val_f1=0.9735 (best=0.9735)


  E13: val_f1=0.9733 (best=0.9735)


  E14: val_f1=0.9729 (best=0.9735)


  E15: val_f1=0.9729 (best=0.9735)
ConvNeXt V2: 0.9735


ConvNeXt V2 probs: (5306, 3)


---
## Phase 3 — Load All Existing Models + Collect Val Probs

In [7]:
existing_models = [
    ('C_focal_v1', 'yuda6_prompt_focal', 'clip', None),
    ('ConvNeXt',   'yuda_convnext_tiny',  'conv', 224),
    ('EffNet',     'yuda2_effnet_b3_300', 'conv', 300),
    ('ResNet50',   'yuda_resnet50',       'conv', 224),
]

all_probs = {}
print("Loading existing models...")
for name, ckpt_name, model_type, img_size in existing_models:
    if model_type == 'clip':
        model = CLIPWithTextFeatures(clip_model, clip_visual, prompts_improved).to(DEVICE)
        loader = clip_val_loader
    else:
        arch = {'ConvNeXt': 'convnext_tiny', 'EffNet': 'efficientnet_b3', 'ResNet50': 'resnet50'}[name]
        model = TrashClassifier(arch, num_classes=3).to(DEVICE)
        loader = conv_val_loader_300 if img_size == 300 else conv_val_loader
    
    ckpt = torch.load(RESULTS / f'{ckpt_name}.pt', map_location=DEVICE, weights_only=True)
    model.load_state_dict(ckpt)
    model.eval()
    
    meta_path = RESULTS / f'{ckpt_name}.json'
    f1_loaded = json.load(open(meta_path)).get('best_val_f1', '?') if meta_path.exists() else '?'
    print(f"  {name}: loaded ({f1_loaded})")
    
    all_probs[name] = get_probs(model, loader)
    del model
    torch.cuda.empty_cache()
    gc.collect()

# Add newly trained models — load from checkpoint
m_cv2 = CLIPWithTextFeatures(clip_model, clip_visual, prompts_improved).to(DEVICE)
m_cv2.load_state_dict(torch.load(RESULTS / 'yuda8_C_focal_v2.pt', map_location=DEVICE, weights_only=True))
m_cv2.eval()
all_probs['C_focal_v2'] = get_probs(m_cv2, clip_val_loader)
del m_cv2
torch.cuda.empty_cache()
gc.collect()

m_conv2 = TrashClassifier('convnextv2_tiny', num_classes=3).to(DEVICE)
m_conv2.load_state_dict(torch.load(RESULTS / 'yuda8_convnextv2_v2.pt', map_location=DEVICE, weights_only=True))
m_conv2.eval()
all_probs['ConvNeXtV2'] = get_probs(m_conv2, conv_val_loader)
del m_conv2
torch.cuda.empty_cache()
gc.collect()

print(f"\nTotal models: {len(all_probs)}")
for name, probs in all_probs.items():
    print(f"  {name}: {probs.shape}")

Loading existing models...
  C_focal_v1: loaded (0.9839496950616109)


  ConvNeXt: loaded (0.974227735498483)


  EffNet: loaded (0.9730807726543511)


  ResNet50: loaded (0.9698417286463156)



Total models: 6
  C_focal_v1: (5306, 3)
  ConvNeXt: (5306, 3)
  EffNet: (5306, 3)
  ResNet50: (5306, 3)
  C_focal_v2: (5306, 3)
  ConvNeXtV2: (5306, 3)


---
## Phase 4 — Stacking Experiments

In [8]:
print("=" * 60)
print("Phase 4: Stacking Experiments")
print("=" * 60)

def run_stacking(name, prob_dict, model_keys):
    X = np.concatenate([prob_dict[k] for k in model_keys], axis=1)
    lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)
    lr.fit(X, val_labels)
    preds = lr.predict(X)
    f1 = f1_score(val_labels, preds, average='macro')
    f1_per = f1_score(val_labels, preds, average=None)
    print(f"  {name:40s} F1={f1:.4f}  [{f1_per[0]:.4f}, {f1_per[1]:.4f}, {f1_per[2]:.4f}]")
    return {'name': name, 'models': '+'.join(k[:10] for k in model_keys), 'best_val_f1': f1, 'f1_per': f1_per, 'meta': lr}

def run_weighted_avg(name, prob_dict, model_keys, weights):
    ensemble = sum(w * prob_dict[k] for w, k in zip(weights, model_keys))
    preds = ensemble.argmax(axis=1)
    f1 = f1_score(val_labels, preds, average='macro')
    f1_per = f1_score(val_labels, preds, average=None)
    print(f"  {name:40s} F1={f1:.4f}  [{f1_per[0]:.4f}, {f1_per[1]:.4f}, {f1_per[2]:.4f}]")
    return {'name': name, 'models': '+'.join(k[:10] for k in model_keys), 'best_val_f1': f1, 'f1_per': f1_per, 'meta': None}

P = all_probs
all_results = []

print("\n--- Weighted Average Baselines ---")
all_results.append(run_weighted_avg('Champion v1 (Cv1+Conv+Eff) 0.70+0.10+0.20',
    P, ['C_focal_v1', 'ConvNeXt', 'EffNet'], [0.70, 0.10, 0.20]))
all_results.append(run_weighted_avg('Champion w/ Cv2 (Cv2+Conv+Eff) 0.70+0.10+0.20',
    P, ['C_focal_v2', 'ConvNeXt', 'EffNet'], [0.70, 0.10, 0.20]))
all_results.append(run_weighted_avg('Champion w/ ConvV2 (Cv1+ConvV2+Eff) 0.70+0.10+0.20',
    P, ['C_focal_v1', 'ConvNeXtV2', 'EffNet'], [0.70, 0.10, 0.20]))
all_results.append(run_weighted_avg('All new (Cv2+ConvV2+Eff) 0.70+0.10+0.20',
    P, ['C_focal_v2', 'ConvNeXtV2', 'EffNet'], [0.70, 0.10, 0.20]))

print("\n--- Stacking Experiments ---")
# S1: Original champion
all_results.append(run_stacking('S1: Cv1+Conv+Eff (original)', P, ['C_focal_v1', 'ConvNeXt', 'EffNet']))
# S2: Champion + ResNet50
all_results.append(run_stacking('S2: Cv1+Conv+Eff+RN50', P, ['C_focal_v1', 'ConvNeXt', 'EffNet', 'ResNet50']))
# S3: New C + old Conv + Eff
all_results.append(run_stacking('S3: Cv2+Conv+Eff', P, ['C_focal_v2', 'ConvNeXt', 'EffNet']))
# S4: Old C + new ConvV2 + Eff
all_results.append(run_stacking('S4: Cv1+ConvV2+Eff', P, ['C_focal_v1', 'ConvNeXtV2', 'EffNet']))
# S5: Both new + Eff
all_results.append(run_stacking('S5: Cv2+ConvV2+Eff', P, ['C_focal_v2', 'ConvNeXtV2', 'EffNet']))
# S6: Both new + Eff + RN50
all_results.append(run_stacking('S6: Cv2+ConvV2+Eff+RN50', P, ['C_focal_v2', 'ConvNeXtV2', 'EffNet', 'ResNet50']))
# S7: All models
all_results.append(run_stacking('S7: All 6 models', P, list(P.keys())))
# S8: Best 3 CNN + both C
all_results.append(run_stacking('S8: Cv1+Cv2+ConvV2+Eff', P, ['C_focal_v1', 'C_focal_v2', 'ConvNeXtV2', 'EffNet']))

df_results = pd.DataFrame(all_results).sort_values('best_val_f1', ascending=False).reset_index(drop=True)
print(f"\n{'=' * 60}")
print("Sorted Results:")
print(f"{'Rank':5s} {'Name':40s} {'F1':8s} {'Per-class':20s}")
print('-' * 75)
for i, (_, r) in enumerate(df_results.iterrows()):
    fp = r.get('f1_per', [0,0,0])
    print(f"{i+1:4d}. {r['name']:40s} {r['best_val_f1']:.4f}   [{fp[0]:.4f} {fp[1]:.4f} {fp[2]:.4f}]")

Phase 4: Stacking Experiments

--- Weighted Average Baselines ---
  Champion v1 (Cv1+Conv+Eff) 0.70+0.10+0.20 F1=0.9856  [0.9795, 0.9930, 0.9843]
  Champion w/ Cv2 (Cv2+Conv+Eff) 0.70+0.10+0.20 F1=0.9860  [0.9791, 0.9949, 0.9841]
  Champion w/ ConvV2 (Cv1+ConvV2+Eff) 0.70+0.10+0.20 F1=0.9856  [0.9796, 0.9930, 0.9843]
  All new (Cv2+ConvV2+Eff) 0.70+0.10+0.20  F1=0.9871  [0.9808, 0.9949, 0.9854]

--- Stacking Experiments ---
  S1: Cv1+Conv+Eff (original)              F1=0.9860  [0.9790, 0.9949, 0.9841]
  S2: Cv1+Conv+Eff+RN50                    F1=0.9853  [0.9788, 0.9937, 0.9835]
  S3: Cv2+Conv+Eff                         F1=0.9859  [0.9790, 0.9949, 0.9837]
  S4: Cv1+ConvV2+Eff                       F1=0.9872  [0.9808, 0.9956, 0.9853]
  S5: Cv2+ConvV2+Eff                       F1=0.9871  [0.9805, 0.9956, 0.9851]
  S6: Cv2+ConvV2+Eff+RN50                  F1=0.9867  [0.9810, 0.9937, 0.9853]
  S7: All 6 models                         F1=0.9872  [0.9813, 0.9949, 0.9855]
  S8: Cv1+Cv2+ConvV

---
## Analysis: Best Model Coefficients

In [9]:
best_result = max(all_results, key=lambda r: r['best_val_f1'])
best_lr = best_result['meta']

print(f"Best approach: {best_result['name']} @ {best_result['best_val_f1']:.4f}")
if best_lr is not None:
    print("\nPer-class model coefficients (|sum|):")
    model_keys = best_result['models'].split('+')
    coef = best_lr.coef_
    for ci, cname in enumerate(CLASS_NAMES):
        print(f"\n  {cname}:")
        for mi, mname in enumerate(model_keys):
            if mi >= coef.shape[1] // 3:
                break
            w = coef[ci, mi*3:(mi+1)*3]
            print(f"    {mname:15s} |sum|={np.abs(w).sum():.4f}  [p0={w[0]:+.2f} p1={w[1]:+.2f} p2={w[2]:+.2f}]")
        print(f"    {'':15s} intercept={best_lr.intercept_[ci]:+.4f}")

best_preds = (best_lr.predict if best_lr else None)
if best_preds is not None:
    preds = best_preds(np.concatenate([P[k] for k in best_result['models'].split('+')], axis=1))
    cm = confusion_matrix(val_labels, preds)
    print(f"\nConfusion Matrix:")
    print(cm)
    print(f"\nNormalized:")
    print((cm.astype(float) / cm.sum(axis=1, keepdims=True)).round(4))

Best approach: S7: All 6 models @ 0.9872

Per-class model coefficients (|sum|):

  Recyclable:
    C_focal_v1      |sum|=3.7874  [p0=+1.91 p1=-0.97 p2=-0.91]
    ConvNeXt        |sum|=0.3507  [p0=+0.10 p1=-0.16 p2=+0.09]
    EffNet          |sum|=0.6828  [p0=+0.35 p1=-0.24 p2=-0.09]
    ResNet50        |sum|=0.7191  [p0=+0.37 p1=-0.12 p2=-0.23]
    C_focal_v2      |sum|=2.2572  [p0=+1.14 p1=-0.71 p2=-0.40]
    ConvNeXtV2      |sum|=1.1521  [p0=+0.59 p1=-0.48 p2=-0.08]
                    intercept=+0.0551

  Electronic:
    C_focal_v1      |sum|=2.6767  [p0=-0.74 p1=+1.29 p2=-0.65]
    ConvNeXt        |sum|=1.2078  [p0=-0.54 p1=+0.55 p2=-0.12]
    EffNet          |sum|=1.7257  [p0=-0.20 p1=+0.81 p2=-0.72]
    ResNet50        |sum|=1.7390  [p0=-0.58 p1=+0.82 p2=-0.34]
    C_focal_v2      |sum|=1.9578  [p0=-0.59 p1=+0.93 p2=-0.44]
    ConvNeXtV2      |sum|=1.7144  [p0=-0.22 p1=+0.80 p2=-0.69]
                    intercept=-0.1829

  Organic:
    C_focal_v1      |sum|=3.0297  [p0=-1.16 p1

---
## Final Summary

In [10]:
print("=" * 60)
print("Yuda 8 — Final Summary")
print("=" * 60)

summary = df_results.copy()
summary.to_csv(RESULTS / 'yuda8_summary.csv', index=False)

print("\nTop 5:")
for i, (_, r) in enumerate(summary.head(5).iterrows()):
    fp = r.get('f1_per', [0,0,0])
    print(f"  {i+1}. {r['name']:45s} F1={r['best_val_f1']:.4f}  [{fp[0]:.4f} {fp[1]:.4f} {fp[2]:.4f}]")

winner = summary.iloc[0]
print(f"\n{'=' * 60}")
print(f"🏆 Champion: {winner['name']} @ {winner['best_val_f1']:.4f}")
print(f"📊 vs yuda_7 (stacking):    0.9860")
print(f"📊 vs yuda_6 (weighted avg): 0.9856")
print(f"📊 Improvement vs yuda_6:    +{winner['best_val_f1'] - 0.9856:.4f}")

# Single model comparison
print(f"\nSingle models:")
for name, probs in P.items():
    f1_single = f1_score(val_labels, probs.argmax(axis=1), average='macro')
    print(f"  {name:20s} F1={f1_single:.4f}")

Yuda 8 — Final Summary

Top 5:
  1. S7: All 6 models                              F1=0.9872  [0.9813 0.9949 0.9855]
  2. S4: Cv1+ConvV2+Eff                            F1=0.9872  [0.9808 0.9956 0.9853]
  3. All new (Cv2+ConvV2+Eff) 0.70+0.10+0.20       F1=0.9871  [0.9808 0.9949 0.9854]
  4. S5: Cv2+ConvV2+Eff                            F1=0.9871  [0.9805 0.9956 0.9851]
  5. S8: Cv1+Cv2+ConvV2+Eff                        F1=0.9868  [0.9803 0.9949 0.9851]

🏆 Champion: S7: All 6 models @ 0.9872
📊 vs yuda_7 (stacking):    0.9860
📊 vs yuda_6 (weighted avg): 0.9856
📊 Improvement vs yuda_6:    +0.0016

Single models:
  C_focal_v1           F1=0.9839
  ConvNeXt             F1=0.9575
  EffNet               F1=0.9701
  ResNet50             F1=0.9674
  C_focal_v2           F1=0.9838
  ConvNeXtV2           F1=0.9729
